# DynaVisR: Benchmark for Visual Reasoning in Dynamic Environments

This notebook expects a Kaggle dataset containing `benchmark.csv` and an `images/` folder.

In [1]:
%pip install pyarrow

Note: you may need to restart the kernel to use updated packages.


In [2]:
from dataclasses import dataclass
import ast
import json
import re
from pathlib import Path
from typing import Any, Sequence

import pandas as pd


OBSTACLE_NAMES = {"A", "B", "C", "D"}
VALID_HIT_NAMES = {"A", "B", "C", "D", "Wall"}


from dataclasses import dataclass, field, asdict, is_dataclass
from typing import Any

@dataclass
class BilliardAnswer:
    hit_object: str = ""
    visible_objects_at_that_moment: list[str] = field(default_factory=list)
    visible_overlapping_objects: list[str] = field(default_factory=list)
    layer_groups_bottom_to_top: list[list[str]] = field(default_factory=list)


def coerce_billiard_answer(value: Any) -> BilliardAnswer:
    if value is None:
        return BilliardAnswer()

    if isinstance(value, BilliardAnswer):
        return BilliardAnswer(
            hit_object=value.hit_object or "",
            visible_objects_at_that_moment=list(value.visible_objects_at_that_moment or []),
            visible_overlapping_objects=list(value.visible_overlapping_objects or []),
            layer_groups_bottom_to_top=[
                list(group or [])
                for group in (value.layer_groups_bottom_to_top or [])
                if group is not None
            ],
        )

    if hasattr(value, "model_dump"):
        value = value.model_dump()
    elif is_dataclass(value):
        value = asdict(value)
    elif not isinstance(value, dict):
        value = {
            "hit_object": getattr(value, "hit_object", ""),
            "visible_objects_at_that_moment": getattr(value, "visible_objects_at_that_moment", []),
            "visible_overlapping_objects": getattr(value, "visible_overlapping_objects", []),
            "layer_groups_bottom_to_top": getattr(value, "layer_groups_bottom_to_top", []),
        }

    return BilliardAnswer(
        hit_object=value.get("hit_object") or "",
        visible_objects_at_that_moment=value.get("visible_objects_at_that_moment") or [],
        visible_overlapping_objects=value.get("visible_overlapping_objects") or [],
        layer_groups_bottom_to_top=value.get("layer_groups_bottom_to_top") or [],
    )


def _maybe_parse_json_like(value: Any) -> Any:
    if isinstance(value, (list, dict)):
        return value
    if value is None:
        return None
    if not isinstance(value, str):
        return value

    text = value.strip()
    if not text:
        return value

    for parser in (json.loads, ast.literal_eval):
        try:
            return parser(text)
        except Exception:
            continue
    return value


def normalize_hit_name(value: Any) -> str:
    if value is None:
        return ""
    text = str(value).strip()
    if not text:
        return ""

    if re.search(r"\bwall\b", text, flags=re.IGNORECASE):
        return "Wall"

    match = re.search(r"\b([A-D])\b", text, flags=re.IGNORECASE)
    if match:
        return match.group(1).upper()

    upper = text.upper()
    if upper in OBSTACLE_NAMES:
        return upper
    if upper == "WALL":
        return "Wall"
    return text


def normalize_name_list(value: Any) -> list[str]:
    value = _maybe_parse_json_like(value)
    if value is None:
        return []
    if isinstance(value, str):
        candidates = re.findall(r"\b(?:[A-D]|Wall)\b", value, flags=re.IGNORECASE)
        if not candidates:
            return []
        value = candidates

    if not isinstance(value, Sequence) or isinstance(value, (bytes, bytearray)):
        value = [value]

    normalized = []
    for item in value:
        name = normalize_hit_name(item)
        if name in VALID_HIT_NAMES and name != "Wall":
            normalized.append(name)

    return sorted(set(normalized))


def normalize_layer_groups(value: Any) -> list[list[str]]:
    value = _maybe_parse_json_like(value)
    if value in (None, ""):
        return []

    if isinstance(value, str):
        value = [value]

    if not isinstance(value, Sequence) or isinstance(value, (bytes, bytearray)):
        return []

    groups: list[list[str]] = []
    for group in value:
        group = _maybe_parse_json_like(group)
        if group is None:
            continue
        if isinstance(group, str):
            group_items = re.findall(r"\b(?:[A-D])\b", group, flags=re.IGNORECASE)
        elif isinstance(group, Sequence):
            group_items = list(group)
        else:
            group_items = [group]

        normalized_group = []
        for item in group_items:
            name = normalize_hit_name(item)
            if name in OBSTACLE_NAMES:
                normalized_group.append(name)

        normalized_group = sorted(set(normalized_group))
        if normalized_group:
            groups.append(normalized_group)

    return groups


def build_structured_prompt(base_prompt: str) -> str:
    return (
        base_prompt.strip()
        + "\n\n"
        + "Return your answer using the provided schema only. "
        + "Use exact object names from this closed set: A, B, C, D, Wall. "
        + "For list fields, return deduplicated names. "
        + "For layer_groups_bottom_to_top, keep the outer list ordered from bottom to top."
    )


def component_scores(
    prediction: BilliardAnswer,
    *,
    q1_hit_object: Any,
    q2_visible_objects: Any,
    q3a_visible_overlapping_objects: Any,
    q3b_layer_groups_bottom_to_top: Any,
) -> dict[str, float]:
    pred_hit = normalize_hit_name(prediction.hit_object)
    gold_hit = normalize_hit_name(q1_hit_object)

    pred_visible = normalize_name_list(prediction.visible_objects_at_that_moment)
    gold_visible = normalize_name_list(q2_visible_objects)

    pred_overlap = normalize_name_list(prediction.visible_overlapping_objects)
    gold_overlap = normalize_name_list(q3a_visible_overlapping_objects)

    pred_layers = normalize_layer_groups(prediction.layer_groups_bottom_to_top)
    gold_layers = normalize_layer_groups(q3b_layer_groups_bottom_to_top)

    return {
        "hit_object": 1.0 if pred_hit == gold_hit else 0.0,
        "visible_objects": 1.0 if pred_visible == gold_visible else 0.0,
        "overlapping_objects": 1.0 if pred_overlap == gold_overlap else 0.0,
        "layer_groups": 1.0 if pred_layers == gold_layers else 0.0,
    }


def total_score(
    prediction: BilliardAnswer,
    *,
    q1_hit_object: Any,
    q2_visible_objects: Any,
    q3a_visible_overlapping_objects: Any,
    q3b_layer_groups_bottom_to_top: Any,
    weights: tuple[float, float, float, float] = (0.40, 0.25, 0.15, 0.20),
    return_components: bool = False,
) -> float | tuple[float, dict[str, float]]:
    scores = component_scores(
        prediction,
        q1_hit_object=q1_hit_object,
        q2_visible_objects=q2_visible_objects,
        q3a_visible_overlapping_objects=q3a_visible_overlapping_objects,
        q3b_layer_groups_bottom_to_top=q3b_layer_groups_bottom_to_top,
    )
    ordered = (
        scores["hit_object"],
        scores["visible_objects"],
        scores["overlapping_objects"],
        scores["layer_groups"],
    )
    total = float(sum(w * s for w, s in zip(weights, ordered)))
    return (total, scores) if return_components else total


def load_eval_csv(csv_path: str | Path, kaggle_dataset_path: str | Path) -> pd.DataFrame:
    csv_path = Path(csv_path)
    dataset_root = Path(kaggle_dataset_path)
    df = pd.read_csv(csv_path)

    required_columns = [
        "prompt",
        "image_rel_path",
        "q1_hit_object",
        "q2_visible_objects",
        "q3a_visible_overlapping_objects",
        "q3b_layer_groups_bottom_to_top",
    ]
    missing = [c for c in required_columns if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required evaluation columns: {missing}")

    df["image_path"] = df["image_rel_path"].map(lambda p: str(dataset_root / str(p)))
    return df


In [3]:
from kaggle_benchmarks import contexts

def _json_compact(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, separators=(",", ":"))

def store_example_outputs(
    *,
    prediction: BilliardAnswer,
    scores: dict[str, float],
    total: float,
) -> None:
    current_run = contexts.get_current().run
    if current_run is None:
        return

    current_run.params.update({
        "pred_hit_object": normalize_hit_name(prediction.hit_object),
        "pred_visible_objects_at_that_moment": _json_compact(
            normalize_name_list(prediction.visible_objects_at_that_moment)
        ),
        "pred_visible_overlapping_objects": _json_compact(
            normalize_name_list(prediction.visible_overlapping_objects)
        ),
        "pred_layer_groups_bottom_to_top": _json_compact(
            normalize_layer_groups(prediction.layer_groups_bottom_to_top)
        ),
        "score_hit_object": scores["hit_object"],
        "score_visible_objects": scores["visible_objects"],
        "score_overlapping_objects": scores["overlapping_objects"],
        "score_layer_groups": scores["layer_groups"],
        "score_total": total,
    })

In [4]:
from pathlib import Path

import kaggle_benchmarks as kbench
from kaggle_benchmarks.content_types import images

KAGGLE_DATASET_PATH = Path('/kaggle/input/datasets/akaliutau/dynavisr')
CSV_PATH = KAGGLE_DATASET_PATH / 'benchmark.csv'

eval_df = load_eval_csv(CSV_PATH, kaggle_dataset_path=KAGGLE_DATASET_PATH)
eval_df = eval_df[
    [
        "prompt",
        "image_path",
        "q1_hit_object",
        "q2_visible_objects",
        "q3a_visible_overlapping_objects",
        "q3b_layer_groups_bottom_to_top",
    ]
]
eval_df.head()

,prompt,image_path,q1_hit_object,q2_visible_objects,q3a_visible_overlapping_objects,q3b_layer_groups_bottom_to_top
0,You are playing the special billiard game. Ini...,/kaggle/input/datasets/akaliutau/dynavisr/imag...,Wall,"[""C"", ""D""]",[],[]
1,You are playing the special billiard game. Ini...,/kaggle/input/datasets/akaliutau/dynavisr/imag...,Wall,"[""A"", ""C""]",[],[]
2,You are playing the special billiard game. Ini...,/kaggle/input/datasets/akaliutau/dynavisr/imag...,Wall,[],[],[]
3,You are playing the special billiard game. Ini...,/kaggle/input/datasets/akaliutau/dynavisr/imag...,D,"[""B"", ""D""]",[],[]
4,You are playing the special billiard game. Ini...,/kaggle/input/datasets/akaliutau/dynavisr/imag...,Wall,"[""B""]",[],[]


In [5]:
@kbench.task(name="dynamic_visual_reasoning")
def dynamic_visual_reasoning(
    llm,
    prompt: str,
    image_path: str,
    q1_hit_object,
    q2_visible_objects,
    q3a_visible_overlapping_objects,
    q3b_layer_groups_bottom_to_top,
) -> float:
    prediction_error = None

    try:
        raw_prediction = llm.prompt(
            build_structured_prompt(prompt),
            image=images.from_path(image_path),
            schema=BilliardAnswer,
        )
    except Exception as e:
        raw_prediction = None
        prediction_error = repr(e)

    prediction = coerce_billiard_answer(raw_prediction)

    total, scores = total_score(
        prediction,
        q1_hit_object=q1_hit_object,
        q2_visible_objects=q2_visible_objects,
        q3a_visible_overlapping_objects=q3a_visible_overlapping_objects,
        q3b_layer_groups_bottom_to_top=q3b_layer_groups_bottom_to_top,
        return_components=True,
    )

    store_example_outputs(prediction=prediction, scores=scores, total=total)

    current_run = contexts.get_current().run
    if current_run is not None and prediction_error is not None:
        current_run.params["prediction_error"] = prediction_error

    return total

In [6]:
@kbench.task(name="Dynamic Visual Reasoning Benchmark")
def dynamic_visual_reasoning_benchmark(llm) -> float:
    runs = dynamic_visual_reasoning.evaluate(
        llm=[llm],
        evaluation_data=eval_df.sample(n=min(100, len(eval_df)), random_state=0),
        n_jobs=4,
    )

    runs_df = runs.as_dataframe().copy()
    runs_df["result_numeric"] = pd.to_numeric(runs_df["result"], errors="coerce")
    valid_scores = runs_df["result_numeric"].dropna()

    mean_score = float(valid_scores.mean()) if len(valid_scores) else 0.0

    current_run = contexts.get_current().run
    if current_run is not None:
        current_run.params.update({
            "n_examples": int(len(runs_df)),
            "n_scored": int(valid_scores.shape[0]),
            "n_failed": int(runs_df["result_numeric"].isna().sum()),
            "mean_score": mean_score,
        })

    return mean_score

In [7]:
benchmark_run = dynamic_visual_reasoning_benchmark.run(kbench.llm)
print(benchmark_run.result)

detail_df = benchmark_run.subruns.as_dataframe().copy()
detail_df["result_numeric"] = pd.to_numeric(detail_df["result"], errors="coerce")
detail_df[detail_df["result_numeric"].isna()][
    ["prompt", "image_path", "result"]
]

detail_df["llm"] = detail_df["llm"].map(lambda x: getattr(x, "name", str(x)))

[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.


1.0


[Parallel(n_jobs=4)]: Done   2 out of   2 | elapsed:  2.7min finished


In [8]:
detail_df.head()


,llm,prompt,image_path,q1_hit_object,q2_visible_objects,q3a_visible_overlapping_objects,q3b_layer_groups_bottom_to_top,pred_hit_object,pred_visible_objects_at_that_moment,pred_visible_overlapping_objects,pred_layer_groups_bottom_to_top,score_hit_object,score_visible_objects,score_overlapping_objects,score_layer_groups,score_total,result,id,result_numeric
run_id,,,,,,,,,,,,,,,,,,,
Run #2,google/gemini-3-flash-preview,You are playing the special billiard game. Ini...,/kaggle/input/datasets/akaliutau/dynavisr/imag...,Wall,[],[],[],Wall,[],[],[],1.0,1.0,1.0,1.0,1.0,1.0,86,1.0
Run #1,google/gemini-3-flash-preview,You are playing the special billiard game. Ini...,/kaggle/input/datasets/akaliutau/dynavisr/imag...,Wall,[],[],[],Wall,[],[],[],1.0,1.0,1.0,1.0,1.0,1.0,26,1.0


In [9]:
import json
from dataclasses import asdict, is_dataclass

def parquet_safe(x):
    if x is None or isinstance(x, (str, int, float, bool)):
        return x
    if is_dataclass(x):
        return json.dumps(asdict(x), ensure_ascii=False)
    if isinstance(x, (list, dict)):
        return json.dumps(x, ensure_ascii=False)
    if hasattr(x, "model_dump"):   # pydantic-style objects
        return json.dumps(x.model_dump(), ensure_ascii=False)
    return str(x)

detail_df = benchmark_run.subruns.as_dataframe().copy()

for col in detail_df.columns:
    if detail_df[col].dtype == "object":
        detail_df[col] = detail_df[col].map(parquet_safe)

detail_df.to_parquet("benchmark_run_details.parquet", index=False)

In [10]:
%choose dynamic_visual_reasoning_benchmark

Kept: Dynamic_Visual_Reasoning_Benchmark-run_id_Run_1_google_gemini-3-flash-preview.run.json
Kept: Dynamic_Visual_Reasoning_Benchmark.task.json
